# 03 — Phase 1 spatial canopy-height model

Phase 1 trains a landscape-specific Attention U-Net from Sentinel-1, Sentinel-2, topographic predictors, and sparse GEDI RH95 supervision. Training switches remain disabled by default to prevent accidental retraining of the frozen production models.

In [ ]:
from __future__ import annotations

import hashlib
import json
import subprocess
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

PROJECT = Path(r"C:\Users\Dell\Desktop\Publication_Clarck\Natural_Sampling")
CONFIG_PATH = PROJECT / "Config" / "pipeline_config.json"
SOURCE_ROOT = PROJECT / "Source" / "Project"
assert PROJECT.is_dir(), PROJECT
assert CONFIG_PATH.is_file(), CONFIG_PATH
PIPELINE = json.loads(CONFIG_PATH.read_text(encoding="utf-8"))

def sha256(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open("rb") as stream:
        for chunk in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

print("Projet :", PROJECT)
print("Configuration :", CONFIG_PATH)
print("Python :", sys.executable)


In [ ]:
EXPECTED_CHANNELS = [
    "S2_LEAFON_B02", "S2_LEAFON_B03", "S2_LEAFON_B04", "S2_LEAFON_B08",
    "S2_LEAFON_B05", "S2_LEAFON_B06", "S2_LEAFON_B07", "S2_LEAFON_B8A",
    "S1_ASC_VV", "S1_ASC_VH", "S1_DESC_VV", "S1_DESC_VH", "AOI_MASK", "DEM", "SLOPE",
]

preflight_rows = []
for site, cfg in PIPELINE["sites"].items():
    catalog = Path(cfg["catalog"])
    experiment = json.loads((catalog / "experiment.json").read_text(encoding="utf-8"))
    order = experiment.get("channel_order") or experiment.get("schema", {}).get("channel_order")
    arrays = sorted(catalog.rglob("*.npy"))
    if not arrays:
        raise FileNotFoundError(f"{site}: no NPY files in {catalog}")
    shape = tuple(np.load(arrays[0], mmap_mode="r", allow_pickle=False).shape)
    required = [catalog / "sample_catalog.csv", catalog / "shot_catalog.csv.gz", catalog / "spatial_split_manifest.csv"]
    missing = [str(path) for path in required if not path.is_file()]
    if missing:
        raise FileNotFoundError(missing)
    if shape != (512, 512, 15):
        raise RuntimeError((site, shape, arrays[0]))
    if list(order) != EXPECTED_CHANNELS:
        raise RuntimeError((site, order))
    preflight_rows.append({"forest": cfg["forest"], "catalog": str(catalog), "shape": shape, "channels": len(order), "status": "PASS"})

pd.DataFrame(preflight_rows)


## Spatial partitions and optimisation

The fixed spatial split is loaded before model fitting. TRAIN is used for optimisation, VAL for model and checkpoint decisions, and TEST only for final held-out evaluation.

In [ ]:
split_rows = []
for site, cfg in PIPELINE["sites"].items():
    manifest = pd.read_csv(Path(cfg["catalog"]) / "spatial_split_manifest.csv")
    split_col = next(name for name in ("split", "subset", "partition") if name in manifest.columns)
    id_col = next(name for name in ("spatial_group_id", "spatial_id", "patch_id", "tile_id", "group_id") if name in manifest.columns)
    memberships = manifest.groupby(id_col)[split_col].nunique()
    leaks = int((memberships > 1).sum())
    if leaks:
        raise RuntimeError(f"{site}: {leaks} spatial units occur in multiple splits")
    counts = manifest[split_col].value_counts().to_dict()
    split_rows.append({"forest": cfg["forest"], "spatial_units": manifest[id_col].nunique(), "leaks": leaks, **counts})
pd.DataFrame(split_rows)


In [ ]:
phase1_protocol = []
for site, cfg in PIPELINE["sites"].items():
    phase1_protocol.append({
        "forest": cfg["forest"], "catalog": cfg["catalog"], "huber_delta": cfg["huber_delta"],
        "validation_domain_m": cfg["primary_validation_domain_m"], "test_domain_m": cfg["primary_test_domain_m"],
        "batch_size": PIPELINE["protocol"]["batch_size"], "seed": PIPELINE["protocol"]["seed"],
    })
pd.DataFrame(phase1_protocol)


## Training and checkpoint provenance

The production workflow trains each landscape from scratch and records the selected Phase 1 checkpoint lineage.

In [ ]:
# Interrupteur unique de reproduction.
REPRODUCE_PHASE1_FROM_SCRATCH = False
RUN_PHASE1 = {site: REPRODUCE_PHASE1_FROM_SCRATCH for site in ("ifran", "maamoura", "agadir")}

TRAIN_SCRIPT = SOURCE_ROOT / "run_b4_c15_train.py"
assert TRAIN_SCRIPT.is_file(), TRAIN_SCRIPT
print(
    "[MODE PHASE 1] "
    + ("REPRODUCTION ACTIVE — training all three landscapes" if REPRODUCE_PHASE1_FROM_SCRATCH
       else "AUDIT ONLY — no training; set REPRODUCE_PHASE1_FROM_SCRATCH=True to reproduce the runs"),
    flush=True,
)

for site, enabled in RUN_PHASE1.items():
    command = [sys.executable, "-u", str(TRAIN_SCRIPT), "--site", site]
    if enabled:
        command.append("--execute")
    print("\n", subprocess.list2cmdline(command), flush=True)
    subprocess.run(command, check=True)


In [ ]:
RUN_PHASE1_EVALUATION = REPRODUCE_PHASE1_FROM_SCRATCH
EVAL_SCRIPT = SOURCE_ROOT / "step07_evaluate_and_plot.py"
for site in PIPELINE["sites"]:
    command = [sys.executable, "-u", str(EVAL_SCRIPT), "--site", site, "--split", "test"]
    if RUN_PHASE1_EVALUATION:
        command.append("--execute-if-missing")
        subprocess.run(command, check=True)
    else:
        print("[EVAL DRY-RUN]", subprocess.list2cmdline(command))


## Held-out diagnostics

The final cells reproduce the Phase 1 TEST scatter diagnostics used to verify scaling and height-range recovery.

In [ ]:
def _column(frame: pd.DataFrame, candidates: tuple[str, ...]) -> str:
    for name in candidates:
        if name in frame.columns:
            return name
    raise KeyError(f"Aucune colonne parmi {candidates}; colonnes={list(frame.columns)}")

def regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict:
    mask = np.isfinite(y_true) & np.isfinite(y_pred)
    y_true, y_pred = y_true[mask], y_pred[mask]
    residual = y_pred - y_true
    slope, intercept = np.polyfit(y_true, y_pred, 1)
    corr = np.corrcoef(y_true, y_pred)[0, 1]
    rmse = float(np.sqrt(np.mean(residual ** 2)))
    r2 = float(1.0 - np.sum(residual ** 2) / np.sum((y_true - y_true.mean()) ** 2))
    std_ratio = float(np.std(y_pred) / np.std(y_true))
    beta = float(y_pred.mean() / y_true.mean())
    kge = float(1.0 - np.sqrt((corr - 1.0) ** 2 + (std_ratio - 1.0) ** 2 + (beta - 1.0) ** 2))
    return {
        "n": len(y_true), "r2": r2, "mae": float(np.mean(np.abs(residual))),
        "rmse": rmse, "bias": float(residual.mean()), "slope": float(slope),
        "intercept": float(intercept), "corr": float(corr),
        "std_ratio": std_ratio, "kge": kge,
    }

def load_prediction_table(path: Path, min_height: float, max_height: float) -> tuple[pd.DataFrame, np.ndarray, np.ndarray]:
    frame = pd.read_csv(path)
    if "aux_shot_uid" in frame.columns:
        order = [name for name in ("abs_temporal_delta_days", "sequence_center_distance", "batch_index", "candidate_order") if name in frame.columns]
        frame = frame.sort_values(order, kind="mergesort") if order else frame
        frame = frame.drop_duplicates("aux_shot_uid", keep="first").copy()
    true_col = _column(frame, ("y_true", "gedi_rh95", "rh95", "target", "observed"))
    pred_col = _column(frame, ("y_pred", "pred_on_growthloss", "prediction_original_coords", "prediction", "predicted"))
    y_true = frame[true_col].to_numpy(float)
    y_pred = frame[pred_col].to_numpy(float)
    keep = np.isfinite(y_true) & np.isfinite(y_pred) & (y_true >= min_height) & (y_true <= max_height)
    return frame.loc[keep].copy(), y_true[keep], y_pred[keep]

def scatter_panel(path: Path, forest: str, domain: tuple[float, float], output: Path) -> dict:
    _, y_true, y_pred = load_prediction_table(path, *domain)
    metrics = regression_metrics(y_true, y_pred)
    fig, ax = plt.subplots(figsize=(7.2, 6.4), constrained_layout=True)
    hb = ax.hexbin(y_true, y_pred, gridsize=48, mincnt=1, bins="log", cmap="viridis")
    low, high = 0.0, float(domain[1])
    ax.plot([low, high], [low, high], "k--", lw=1.5, label="1:1")
    x = np.linspace(low, high, 100)
    ax.plot(x, metrics["intercept"] + metrics["slope"] * x, color="crimson", lw=1.8, label="Regression")
    ax.set(xlim=(low, high), ylim=(low, high), xlabel="Observed GEDI RH95 (m)", ylabel="Predicted height (m)")
    ax.set_title(f"{forest} — TEST unique-nearest — RH95 {domain[0]:g}–{domain[1]:g} m", fontweight="bold")
    text = "\n".join([
        f"n = {metrics['n']}", f"R² = {metrics['r2']:.4f}", f"MAE = {metrics['mae']:.4f} m",
        f"RMSE = {metrics['rmse']:.4f} m", f"Slope = {metrics['slope']:.4f}",
        f"Std ratio = {metrics['std_ratio']:.4f}", f"Bias = {metrics['bias']:+.4f} m",
        f"KGE = {metrics['kge']:.4f}",
    ])
    ax.text(0.025, 0.975, text, transform=ax.transAxes, va="top", bbox=dict(facecolor="white", alpha=.9))
    ax.legend(loc="lower right")
    fig.colorbar(hb, ax=ax, label="log10(N)")
    output.parent.mkdir(parents=True, exist_ok=True)
    fig.savefig(output, dpi=600, bbox_inches="tight")
    fig.savefig(output.with_suffix(".pdf"), bbox_inches="tight")
    plt.show()
    return metrics


In [ ]:
phase1_metrics = []
for site, cfg in PIPELINE["sites"].items():
    path = Path(cfg["test_predictions"])
    if not path.is_file():
        print("[MISSING — lancer STEP07]", site, path)
        continue
    domain = tuple(map(float, cfg["primary_test_domain_m"]))
    output = PROJECT / "Results" / "Final_Article" / "Phase1" / cfg["forest"] / "scatter_test.png"
    metrics = scatter_panel(path, cfg["forest"], domain, output)
    phase1_metrics.append({"forest": cfg["forest"], **metrics})
phase1_metrics = pd.DataFrame(phase1_metrics)
phase1_metrics
